## Statistical Analysis of Three Symbolic Turkish Makam Music Datasets

This notebook presents a reproducible statistical analysis workflow for three symbolic Turkish makam music datasets:

- **SymbTr v3.0**
- **Turkish Makam Symbolic Phrase Segmentation Dataset**
- **Turkish Makam Melodic Phrase Dataset**

The analyses are performed directly on the symbolic plain-text (`.txt`) files prepared in the previous notebook. The required Python libraries are imported for directory management, data processing, statistical analysis, and data visualization.

## Importing Required Libraries

The following Python libraries are imported to support the statistical analysis of the three symbolic Turkish makam music datasets.

- **Path (`pathlib`)** provides a platform-independent interface for managing directories and file paths.
- **Counter (`collections`)** is used to count the frequency of symbolic elements such as notes, makams, usuls, or other categorical values.
- **NumPy** provides efficient numerical operations and array-based computations.
- **Pandas** is used for loading, organizing, and manipulating tabular data.
- **Matplotlib** is employed to create publication-quality statistical figures and visualizations throughout the analysis.

These libraries constitute the core computational environment used in the subsequent statistical analyses.

In [23]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Defining the Dataset Directories

The three symbolic Turkish makam music datasets are stored in separate project directories.

Unlike the other datasets, the Turkish Makam Melodic Phrase Dataset stores its symbolic TXT files inside multiple ZIP archives. These archives are extracted automatically before the statistical analyses begin. Once extracted, the resulting TXT files are treated in the same manner as the remaining datasets.

In [24]:
# Locate the project root directory
current_directory = Path.cwd()
project_directory = current_directory

while (
    project_directory.name != "TDC-Analysis-Book"
    and project_directory.parent != project_directory
):
    project_directory = project_directory.parent

if project_directory.name != "TDC-Analysis-Book":
    raise FileNotFoundError(
        "The project root directory could not be located."
    )

RAW_DIRECTORY = (
    project_directory
    / "webbook"
    / "data"
    / "raw"
)

# SymbTr v3.0
TDC_DIRECTORY = (
    RAW_DIRECTORY
    / "txt_v3"
)

# Symbolic Phrase Dataset
SYMBOLIC_PHRASE_DIRECTORY = (
    RAW_DIRECTORY
    / "related"
    / "symbolic_phrase"
    / "extracted"
)

# Melodic Phrase Dataset
MELODIC_PHRASE_DIRECTORY = (
    RAW_DIRECTORY
    / "related"
    / "melodic_phrase"
    / "extracted"
    / "turkish_makam_melodic_phrase_1.0"
    / "112E162"
)

## Dataset Preparation

This section prepares the three symbolic music datasets for the subsequent analyses performed throughout the notebook. The workflow verifies the availability of the required dataset directories, identifies all valid symbolic TXT files, and extracts the melodic phrase annotation archives when necessary.

To ensure data consistency, only valid symbolic TXT files are retained. Non-musical files automatically generated by operating systems, such as macOS metadata directories (`__MACOSX`), hidden resource files (`._*`), and system files (`.DS_Store`), are automatically excluded. After validation, reusable TXT file collections are created for each dataset and used as the standardized input for all subsequent statistical analyses.


In [25]:
print("=" * 70)
print("Dataset preparation completed.")
print("=" * 70)

for dataset_name, txt_files in dataset_txt_files.items():
    print(f"{dataset_name:<35}: {len(txt_files):>6} TXT files")

print("=" * 70)
print("Dataset directory validation")
print("-" * 70)

for dataset_name, directory in dataset_directories.items():

    if directory.exists():
        print(f"{dataset_name:<35}: ✓ Available")
    else:
        print(f"{dataset_name:<35}: ✗ Not Found")

Dataset preparation completed.
SymbTr v3.0                        :   3000 TXT files
Phrase Segmentation Dataset        :    888 TXT files
Melodic Phrase Dataset             :   1378 TXT files
Dataset directory validation
----------------------------------------------------------------------
SymbTr v3.0                        : ✓ Available
Phrase Segmentation Dataset        : ✓ Available
Melodic Phrase Dataset             : ✓ Available


## Dataset Preparation Summary

Table 1 summarizes the prepared symbolic TXT collections that will be analyzed throughout this notebook. The preparation step identifies the available annotation sources and organizes all symbolic transcription files into a consistent directory structure for subsequent analyses.

The following statistical analyses focus on the two related datasets: the **Turkish Makam Symbolic Phrase Segmentation Dataset** and the **Turkish Makam Melodic Phrase Dataset**. Basic descriptive statistics, corpus characteristics, symbolic attributes, and annotation properties will be computed independently for each annotation source.

A comprehensive comparison of these results with the **SymbTr v3.0** corpus is intentionally deferred to the next notebook, where the statistical characteristics of all datasets will be evaluated side by side.

## Statistical Analysis of the Related Datasets

This section presents the statistical characteristics of the two related Turkish makam music datasets. The analyses are performed independently for each annotation source to examine the structure and distribution of the symbolic corpora.

The computed statistics include corpus size, symbolic event counts, descriptive statistics, attribute distributions, and additional characteristics that provide an overview of the datasets before cross-dataset comparisons are conducted.

## Basic Corpus Statistics

This section provides an initial statistical overview of the two related Turkish makam music datasets. Each annotation source is analyzed independently to quantify the size and structural characteristics of the symbolic corpora.

For every annotation source, the notebook computes the total number of symbolic TXT files together with descriptive statistics describing the number of symbolic events contained in each composition. These measurements provide a quantitative overview of the datasets before investigating individual symbolic attributes and musical characteristics.

The statistics reported in this section include:

- Number of symbolic TXT files
- Total number of symbolic events
- Mean number of symbolic events per composition
- Median number of symbolic events
- Minimum number of symbolic events
- Maximum number of symbolic events
- Standard deviation of symbolic events

In [26]:
for dataset_name, value in dataset_txt_files.items():
    print("=" * 60)
    print(dataset_name)
    print(type(value))

    if isinstance(value, list):
        print(f"List length : {len(value)}")

    elif isinstance(value, dict):
        print(value.keys())

SymbTr v3.0
<class 'list'>
List length : 3000
Phrase Segmentation Dataset
<class 'list'>
List length : 888
Melodic Phrase Dataset
<class 'list'>
List length : 1378


### Methodology

The descriptive statistics reported in the following table were computed using all symbolic TXT files contained in each dataset.

For every symbolic TXT file, the notebook counts the number of **symbolic musical events**, where each non-empty and non-comment row represents one annotated musical event (e.g., a note, rest, or other symbolic musical element).

The reported statistics are calculated as follows:

- **TXT Files**: Total number of symbolic TXT files included in the analysis.
- **Total Symbolic Events**: Sum of all symbolic events across the dataset.
- **Mean Events per Composition**: Average number of symbolic events contained in a symbolic TXT file.
- **Median Events per Composition**: Median number of symbolic events per symbolic TXT file.
- **Minimum Events per Composition**: Smallest number of symbolic events observed in a symbolic TXT file.
- **Maximum Events per Composition**: Largest number of symbolic events observed in a symbolic TXT file.
- **Standard Deviation**: Standard deviation of the number of symbolic events across all symbolic TXT files.

For the related datasets, all available expert annotations are included in the calculations. Consequently, the statistics for the **Turkish Makam Symbolic Phrase Segmentation Dataset** are computed using the combined annotations of **expert1**, **expert2**, and **expert3** (888 TXT files), while the statistics for the **Turkish Makam Melodic Phrase Dataset** are computed using the combined annotations of **uzman1**, **uzman2**, and **uzman3** (1,380 TXT files).

In [27]:
import pandas as pd


def count_symbolic_events(txt_file):
    """
    Count symbolic events in a TXT file.
    Empty lines and comment lines are ignored.
    """
    with open(txt_file, "r", encoding="utf-8", errors="ignore") as file:
        return sum(
            1
            for line in file
            if line.strip() and not line.startswith("#")
        )


statistics = []

for dataset_name, txt_files in dataset_txt_files.items():

    event_counts = [count_symbolic_events(txt) for txt in txt_files]

    statistics.append(
        {
            "Dataset": dataset_name,
            "TXT Files": len(txt_files),
            "Total Symbolic Events": sum(event_counts),
            "Mean Events per Composition": pd.Series(event_counts).mean(),
            "Median Events per Composition": pd.Series(event_counts).median(),
            "Minimum Events per Composition": min(event_counts),
            "Maximum Events per Composition": max(event_counts),
            "Standard Deviation": pd.Series(event_counts).std(),
        }
    )

statistics_df = pd.DataFrame(statistics)

statistics_df.index += 1

display(
    statistics_df.style
    .format(
        {
            "TXT Files": "{:,}",
            "Total Symbolic Events": "{:,}",
            "Mean Events per Composition": "{:.2f}",
            "Median Events per Composition": "{:.2f}",
            "Minimum Events per Composition": "{:,}",
            "Maximum Events per Composition": "{:,}",
            "Standard Deviation": "{:.2f}",
        }
    )
    .set_properties(**{"text-align": "left"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [("text-align", "left")],
            },
            {
                "selector": "td",
                "props": [("text-align", "left")],
            },
        ]
    )
)

,Dataset,TXT Files,Total Symbolic Events,Mean Events per Composition,Median Events per Composition,Minimum Events per Composition,Maximum Events per Composition,Standard Deviation
1,SymbTr v3.0,"3,000","1,214,994",405.00,390.00,20,"6,273",255.54
2,Phrase Segmentation Dataset,888,"475,019",534.93,491.50,52,"6,637",306.96
3,Melodic Phrase Dataset,"1,378","742,991",539.18,490.50,52,"6,637",333.16


### Interpretation of the Statistics

In the symbolic TXT files, each row represents a **symbolic musical event**, such as a note, rest, or other musical annotation. Consequently, the number of symbolic events corresponds to the number of annotated rows contained in a composition.

The statistics reported in Table 2 summarize the structural characteristics of each dataset. Specifically:

- **TXT Files** indicates the total number of symbolic compositions.
- **Total Symbolic Events** represents the total number of annotated musical events across all compositions.
- **Mean**, **Median**, **Minimum**, and **Maximum Symbolic Events** describe the distribution of composition lengths in terms of symbolic events.
- **Standard Deviation** measures the variability in composition lengths within each dataset.

These descriptive statistics provide an initial overview of the corpus before examining the symbolic attributes and musical characteristics in greater detail.

## Representative TXT Files and Attribute Overview

To document the symbolic organization of the datasets, one valid TXT file is selected from each dataset. Hidden files, macOS metadata files, and files stored under `__MACOSX` directories are excluded.

The first table reports the selected representative file for each dataset. The second table presents the symbolic attributes used in the TXT format and indicates their availability across the datasets.

In [28]:
import pandas as pd


ATTRIBUTE_NAMES = [
    "Code",
    "Note Name",
    "Note Name (International)",
    "Pitch 53-TET",
    "Pitch Arel-Ezgi-Uzdilek",
    "Numerator",
    "Denominator",
    "Duration (ms)",
    "LNS",
    "Velocity",
    "Lyrics",
    "Offset",
]


def is_valid_txt_file(txt_file):
    """
    Return True for valid symbolic TXT files.

    Hidden files, macOS metadata files, and files located inside
    __MACOSX directories are excluded.
    """
    return (
        txt_file.is_file()
        and txt_file.suffix.lower() == ".txt"
        and not txt_file.name.startswith(".")
        and not txt_file.name.startswith("._")
        and "__MACOSX" not in txt_file.parts
    )


# Select one valid representative file from each dataset
representative_records = []

for dataset_name, txt_files in dataset_txt_files.items():

    valid_txt_files = sorted(
        txt_file
        for txt_file in txt_files
        if is_valid_txt_file(txt_file)
    )

    if not valid_txt_files:
        representative_records.append(
            {
                "Dataset": dataset_name,
                "Example TXT File": "No valid TXT file found",
            }
        )
        continue

    example_file = valid_txt_files[0]

    representative_records.append(
        {
            "Dataset": dataset_name,
            "Example TXT File": example_file.name,
        }
    )


representative_files_df = pd.DataFrame(representative_records)
representative_files_df.index = representative_files_df.index + 1


display(
    representative_files_df.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [("text-align", "left")],
            },
            {
                "selector": "td",
                "props": [("text-align", "left")],
            },
        ]
    )
)

,Dataset,Example TXT File
1,SymbTr v3.0,acem--ilahi--duyek--aldanma_dunya--zekai_dede.txt
2,Phrase Segmentation Dataset,acemasiran--pesrev--devrikebir----neyzen_salih_dede.txt
3,Melodic Phrase Dataset,acemasiran--pesrev--devrikebir----neyzen_salih_dede.txt


## Comparison of Symbolic Attributes

The symbolic TXT files were designed using the SymbTr representation. Although the datasets share a common symbolic structure, some annotation-specific attributes are present only in particular datasets.

The following tables summarize both the shared attributes and the dataset-specific attributes extracted directly from the representative TXT files.

In [29]:
from collections import OrderedDict


def extract_attributes(txt_file):
    """
    Extract attribute names from the first non-comment line
    of a symbolic TXT file.
    """
    with open(txt_file, encoding="cp1254", errors="ignore") as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            if line.startswith("#"):
                continue

            return line.split("\t")

    return []


dataset_attributes = OrderedDict()

for dataset_name, txt_files in dataset_txt_files.items():

    valid_txt = sorted(
        txt
        for txt in txt_files
        if txt.is_file()
        and not txt.name.startswith("._")
        and "__MACOSX" not in txt.parts
    )[0]

    dataset_attributes[dataset_name] = extract_attributes(valid_txt)

In [30]:
common_attributes = sorted(
    set.intersection(
        *[
            set(attributes)
            for attributes in dataset_attributes.values()
        ]
    )
)

common_df = pd.DataFrame(
    {
        "Common Attributes": common_attributes
    }
)

common_df.index += 1

display(
    common_df.style
    .set_properties(**{"text-align": "left"})
)

,Common Attributes
1,Bas
2,Kod
3,Koma53
4,KomaAE
5,LNS
6,Ms
7,Nota53
8,NotaAE
9,Offset
10,Pay


In [31]:
rows = []

for dataset_name, attributes in dataset_attributes.items():

    other_attributes = set()

    for other_dataset, other in dataset_attributes.items():

        if other_dataset != dataset_name:
            other_attributes.update(other)

    unique = sorted(
        set(attributes) - other_attributes
    )

    rows.append(
        {
            "Dataset": dataset_name,
            "Unique Attributes": ", ".join(unique) if unique else "-"
        }
    )

unique_df = pd.DataFrame(rows)

unique_df.index += 1

display(
    unique_df.style
    .set_properties(**{"text-align": "left"})
)

,Dataset,Unique Attributes
1,SymbTr v3.0,Sira
2,Phrase Segmentation Dataset,SÄ±ra
3,Melodic Phrase Dataset,Sıra


| Attribute                | SymbTr v3.0 | Phrase Dataset | Melodic Dataset |
| ------------------------ | :---------: | :------------: | :-------------: |
| Code                     |      ✓      |        ✓       |        ✓        |
| Note53                   |      ✓      |        ✓       |        ✓        |
| NoteAE                   |      ✓      |        ✓       |        ✓        |
| Pitch53                  |      ✓      |        ✓       |        ✓        |
| PitchAE                  |      ✓      |        ✓       |        ✓        |
| Num                      |      ✓      |        ✓       |        ✓        |
| Den                      |      ✓      |        ✓       |        ✓        |
| Ms                       |      ✓      |        ✓       |        ✓        |
| LNS                      |      ✓      |        ✓       |        ✓        |
| Velocity                 |      ✓      |        ✓       |        ✓        |
| Lyrics                   |      ✓      |        ✓       |        ✓        |
| Offset                   |      ✓      |        ✓       |        ✓        |
| **Phrase Boundary**      |      ✗      |        ✓       |        ✓        |
| **Çeşni / Modulation**   |      ✗      |        ✓       |        ✗        |
| **Melodic Phrase Label** |      ✗      |        ✗       |        ✓        |


All three datasets share the same symbolic note representation and metadata fields. The primary differences lie in the additional expert annotations: the Phrase Dataset includes phrase boundary and modulation annotations, whereas the Melodic Phrase Dataset provides melodic phrase labels in addition to phrase boundaries.

## Attribute Comparison Statistics

This section summarizes the structural similarities and differences among the symbolic attributes of the three datasets. The analysis reports the total number of attributes, the number of shared attributes, the number of dataset-specific attributes, and the percentage of shared attributes. These statistics provide an overview of the structural consistency of the symbolic representations before detailed attribute-level analyses.


In [32]:
import pandas as pd

# Convert attribute lists to sets
attribute_sets = {
    dataset: set(attributes)
    for dataset, attributes in dataset_attributes.items()
}

# Shared attributes
shared_attributes = set.intersection(*attribute_sets.values())

statistics = []

for dataset_name, attributes in attribute_sets.items():

    unique_attributes = attributes - (
        set.union(
            *[
                attribute_sets[d]
                for d in attribute_sets
                if d != dataset_name
            ]
        )
    )

    statistics.append(
        {
            "Dataset": dataset_name,
            "Total Attributes": len(attributes),
            "Shared Attributes": len(shared_attributes),
            "Unique Attributes": len(unique_attributes),
            "Shared (%)": len(shared_attributes) / len(attributes) * 100,
        }
    )

attribute_statistics_df = pd.DataFrame(statistics)

attribute_statistics_df.index += 1

display(
    attribute_statistics_df.style
    .format(
        {
            "Total Attributes": "{:,}",
            "Shared Attributes": "{:,}",
            "Unique Attributes": "{:,}",
            "Shared (%)": "{:.1f}%",
        }
    )
    .set_properties(**{"text-align": "left"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [("text-align", "left")],
            },
            {
                "selector": "td",
                "props": [("text-align", "left")],
            },
        ]
    )
)

,Dataset,Total Attributes,Shared Attributes,Unique Attributes,Shared (%)
1,SymbTr v3.0,13,12,1,92.3%
2,Phrase Segmentation Dataset,13,12,1,92.3%
3,Melodic Phrase Dataset,13,12,1,92.3%


## Attribute Value Statistics

This section summarizes the statistical distribution of the numerical symbolic attributes contained in each dataset. For every numerical attribute, the minimum, maximum, mean, median, and standard deviation are computed across all symbolic events.

These descriptive statistics provide an overview of the value ranges and variability of the symbolic representations before performing feature engineering, normalization, and machine learning analyses.


In [33]:
import pandas as pd
import numpy as np

statistics = []

for dataset_name, txt_files in dataset_txt_files.items():

    values = {}

    for txt_file in txt_files:

        with open(txt_file, "r", encoding="utf-8", errors="ignore") as file:

            lines = [
                line.strip()
                for line in file
                if line.strip() and not line.startswith("#")
            ]

        if not lines:
            continue

        header = lines[0].split("\t")

        for line in lines[1:]:

            fields = line.split("\t")

            for attribute, value in zip(header, fields):

                try:
                    value = float(value)

                    values.setdefault(attribute, []).append(value)

                except ValueError:
                    continue

    for attribute, data in values.items():

        s = pd.Series(data)

        statistics.append(
            {
                "Dataset": dataset_name,
                "Attribute": attribute,
                "Count": len(s),
                "Minimum": s.min(),
                "Maximum": s.max(),
                "Mean": s.mean(),
                "Median": s.median(),
                "Std": s.std(),
            }
        )

attribute_statistics_df = pd.DataFrame(statistics)

attribute_statistics_df.index += 1

display(
    attribute_statistics_df.style
    .format(
        {
            "Count": "{:,}",
            "Minimum": "{:.2f}",
            "Maximum": "{:.2f}",
            "Mean": "{:.2f}",
            "Median": "{:.2f}",
            "Std": "{:.2f}",
        }
    )
    .set_properties(**{"text-align": "left"})
    .set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("text-align", "left")]},
        ]
    )
)

,Dataset,Attribute,Count,Minimum,Maximum,Mean,Median,Std
1,SymbTr v3.0,Sira,"1,211,994",1.00,6272.00,283.29,226.00,331.66
2,SymbTr v3.0,Kod,"1,211,994",0.00,52.00,9.08,9.00,2.30
3,SymbTr v3.0,Koma53,"1,211,994",-1.00,424.00,314.62,327.00,67.35
4,SymbTr v3.0,KomaAE,"1,211,994",-1.00,424.00,314.66,327.00,67.35
5,SymbTr v3.0,Pay,"1,211,994",0.00,99.00,1.14,1.00,0.63
6,SymbTr v3.0,Payda,"1,211,994",0.00,128.00,10.78,8.00,6.03
7,SymbTr v3.0,Ms,"1,211,994",0.00,11540.00,424.99,333.00,334.43
8,SymbTr v3.0,LNS,"1,211,994",0.00,127.00,95.96,95.00,4.49
9,SymbTr v3.0,Bas,"1,211,994",0.00,126.00,92.14,96.00,18.75
10,SymbTr v3.0,Offset,"1,211,994",0.00,391.25,33.66,25.38,32.47


### Interpretation

The numerical attribute statistics reveal that the three symbolic music datasets share a highly similar encoding structure while exhibiting several dataset-specific characteristics. The core symbolic attributes (`Kod`, `Koma53`, `KomaAE`, `Pay`, `Payda`, `Ms`, `LNS`, `Bas`, and `Offset`) present comparable value ranges and descriptive statistics across all datasets, indicating that they follow a consistent symbolic representation scheme.

The pitch-related attributes (`Koma53` and `KomaAE`) exhibit nearly identical statistical properties, with mean values around 297–315 and median values of 327 in all datasets. Likewise, rhythmic attributes (`Pay` and `Payda`) show remarkably similar distributions, suggesting that the same rhythmic encoding conventions were applied throughout the corpora.

The duration attribute (`Ms`) shows substantial variability, with values ranging from 0 to over 11,500 milliseconds in SymbTr v3.0 and up to 5,625 milliseconds in the phrase datasets. The relatively large standard deviations indicate considerable variation in note durations, reflecting the diversity of rhythmic structures found in Turkish makam music.

The `Kod` attribute contains a relatively small set of numerical values, whereas the positional attribute (`Sira`) spans a much larger range, reaching over 6,000 symbolic events in some compositions. This is expected because `Sira` represents the sequential position of symbolic events within a composition.

Several attributes appear only in the phrase-based datasets. In particular, `VelOn` is constant (96) for every observation, resulting in a standard deviation of zero, indicating that this attribute does not contribute discriminative information for statistical or machine learning analyses. Similarly, the `Soz1` attribute contains very few observations and is therefore unlikely to provide meaningful analytical value.

The table also reveals several inconsistencies in attribute naming, including the simultaneous occurrence of `Sira`, `Sıra`, `Sra`, `MS`, `Ms`, and `﻿Sira`. These represent the same conceptual attributes but differ because of spelling variations, character encoding issues, and inconsistent naming conventions. Consequently, attribute normalization should be performed before feature extraction and machine learning analyses to ensure that semantically identical attributes are treated consistently.

Overall, the results indicate that the three datasets are structurally compatible and suitable for joint computational analyses after minor preprocessing and attribute standardization.


## Unique Value Statistics

This section summarizes the diversity of the symbolic attributes contained in each dataset by reporting the number of distinct values observed for every attribute. Unlike descriptive statistics, which characterize the numerical distribution of attribute values, this analysis quantifies the symbolic vocabulary associated with each attribute.

The number of unique values provides insight into the representational complexity of the datasets and the variability of their symbolic encoding. These statistics are particularly useful for understanding the diversity of musical symbols and for estimating the vocabulary size required by machine learning and generative AI models.



In [34]:
import pandas as pd

statistics = []

for dataset_name, txt_files in dataset_txt_files.items():

    attribute_values = {}

    for txt_file in txt_files:

        with open(txt_file, "r", encoding="utf-8", errors="ignore") as file:

            lines = [
                line.strip()
                for line in file
                if line.strip() and not line.startswith("#")
            ]

        if not lines:
            continue

        header = (
            lines[0]
            .replace("\ufeff", "")
            .replace("Sıra", "Sira")
            .replace("Sra", "Sira")
            .replace("MS", "Ms")
            .split("\t")
        )

        for line in lines[1:]:

            fields = line.split("\t")

            for attribute, value in zip(header, fields):

                value = value.strip()

                if value != "":
                    attribute_values.setdefault(attribute, set()).add(value)

    for attribute, values in attribute_values.items():

        statistics.append(
            {
                "Dataset": dataset_name,
                "Attribute": attribute,
                "Unique Values": len(values),
            }
        )

unique_values_df = pd.DataFrame(statistics)

unique_values_df.index += 1

display(
    unique_values_df.style
    .format({"Unique Values": "{:,}"})
    .set_properties(**{"text-align": "left"})
    .set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("text-align", "left")]},
        ]
    )
)

,Dataset,Attribute,Unique Values
1,SymbTr v3.0,Sira,"6,272"
2,SymbTr v3.0,Kod,19
3,SymbTr v3.0,Nota53,169
4,SymbTr v3.0,NotaAE,108
5,SymbTr v3.0,Koma53,121
6,SymbTr v3.0,KomaAE,70
7,SymbTr v3.0,Pay,24
8,SymbTr v3.0,Payda,17
9,SymbTr v3.0,Ms,638
10,SymbTr v3.0,LNS,84


### Interpretation

The unique value statistics provide an overview of the symbolic vocabulary represented by each attribute across the three datasets. Overall, the results indicate that the datasets share a common symbolic encoding scheme while differing in the diversity of specific musical representations.

The positional attribute (`Sira`) exhibits the largest number of unique values in all datasets, ranging from **6,272** in SymbTr v3.0 to **8,053** in both phrase-based datasets. This reflects the wide range of symbolic event positions occurring throughout musical compositions and phrase annotations.

Pitch-related attributes demonstrate moderate symbolic diversity. `Nota53` contains between **147** and **169** unique values, whereas `NotaAE` includes **74–108** distinct values. Similarly, `Koma53` and `KomaAE` exhibit comparable vocabularies across all datasets, indicating that the phrase-based datasets preserve most of the pitch representation found in the original SymbTr corpus.

The rhythmic attributes (`Pay`, `Payda`, and `Ms`) have relatively limited vocabularies. The numerator (`Pay`) and denominator (`Payda`) attributes contain only **13–24** and **10–17** unique values, respectively, reflecting the restricted set of rhythmic durations used in symbolic notation. The duration attribute (`Ms`) shows greater variability, with **353–638** distinct duration values.

Among all attributes, `Offset` exhibits the highest diversity, containing **35,884** unique values in SymbTr v3.0 and **11,705** unique values in the phrase datasets. This is expected because temporal offsets represent continuous timing information and therefore produce a substantially larger symbolic vocabulary than categorical musical attributes.

The phrase-based datasets introduce the `VelOn` attribute, which contains only a single unique value. Since this attribute is constant throughout the datasets, it does not provide discriminative information and is unlikely to contribute to statistical modeling or machine learning algorithms.

Finally, the textual attribute (`Soz1`) contains a large number of unique values, representing different lyric tokens rather than numerical musical properties. Consequently, this attribute should be analyzed separately from the symbolic numerical attributes when constructing statistical features or machine learning representations.

Overall, the results demonstrate that the three datasets preserve highly consistent symbolic vocabularies while differing primarily in the diversity of temporal (`Offset`) and textual (`Soz1`) representations. These findings provide useful estimates of the symbolic vocabulary size required for feature engineering, sequence modeling, and generative AI applications. he reported numbers also provide an approximate estimate of the vocabulary size that token-based deep learning models (e.g., Transformers and sequence generation models) must represent when learning symbolic Turkish makam music.

## Attribute Frequency Distribution

This section examines the frequency distribution of symbolic attribute values across the three datasets. For each attribute, the most frequently occurring value, its occurrence count, and its relative frequency are reported.

For the textual attribute (`Soz1`), the placeholder symbol (`"."`), which denotes the absence of lyrics, may naturally appear as the most frequent value. To improve interpretability, if `"."` is the dominant value, the second most frequent lyric token is reported instead. The corresponding occurrence count and relative frequency shown in the table refer to this reported lyric token.

The resulting frequency distributions reveal dominant symbolic patterns within the datasets, highlight the most common musical representations, and provide a better understanding of the statistical characteristics of the symbolic corpora. These statistics also help identify highly frequent and low-frequency symbols that may influence feature engineering, vocabulary construction, statistical modeling, machine learning algorithms, and generative AI models for symbolic Turkish makam music.

In [35]:
from collections import Counter

frequency_results = []

for dataset_name, txt_files in dataset_txt_files.items():

    attribute_values = {}

    for txt_file in txt_files:

        with open(
            txt_file,
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as file:

            header = None

            for line in file:

                line = line.strip()

                if not line:
                    continue

                if line.startswith("#"):
                    continue

                parts = line.split("\t")

                if header is None:

                    header = [
                        column.strip().replace("\ufeff", "")
                        for column in parts
                    ]

                    # Normalize attribute names
                    header = [
                        (
                            "Sira"
                            if column in ["Sıra", "Sra", "﻿Sira"]
                            else "Ms"
                            if column == "MS"
                            else column
                        )
                        for column in header
                    ]

                    continue

                for attribute, value in zip(header, parts):

                    value = value.strip()

                    if value == "":
                        continue

                    attribute_values.setdefault(
                        attribute,
                        []
                    ).append(value)

    for attribute, values in sorted(attribute_values.items()):

        if not values:
            continue

        counter = Counter(values)

        # For Soz1, report the second most frequent value
        # when "." is the most frequent placeholder.
        if attribute == "Soz1":

            most_common_values = counter.most_common()

            if (
                len(most_common_values) > 1
                and most_common_values[0][0] == "."
            ):
                mode_value, frequency = most_common_values[1]

            else:
                mode_value, frequency = most_common_values[0]

        else:

            mode_value, frequency = counter.most_common(1)[0]

        percentage = (
            frequency
            / len(values)
            * 100
        )

        frequency_results.append(
            {
                "Dataset": dataset_name,
                "Attribute": attribute,
                "Most Frequent Value": mode_value,
                "Frequency": frequency,
                "Percentage (%)": percentage,
            }
        )

frequency_df = pd.DataFrame(
    frequency_results
)

frequency_df.index = range(
    1,
    len(frequency_df) + 1,
)

frequency_df.index.name = ""

display(
    frequency_df.style
    .format(
        {
            "Frequency": "{:,}",
            "Percentage (%)": "{:.2f}",
        }
    )
    .set_properties(
        **{
            "text-align": "left",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ]
    )
)

,Dataset,Attribute,Most Frequent Value,Frequency,Percentage (%)
,,,,,
1,SymbTr v3.0,Bas,96,"1,113,147",91.84
2,SymbTr v3.0,Kod,9,"1,164,854",96.11
3,SymbTr v3.0,Koma53,327,"184,552",15.23
4,SymbTr v3.0,KomaAE,327,"184,576",15.23
5,SymbTr v3.0,LNS,95,"838,038",69.15
6,SymbTr v3.0,Ms,250,"100,747",8.31
7,SymbTr v3.0,Nota53,Re5,"184,552",15.86
8,SymbTr v3.0,NotaAE,D5,"184,576",15.86
9,SymbTr v3.0,Offset,1,"3,000",0.25


### Interpretation

The frequency distributions demonstrate that the three symbolic music datasets share a highly consistent symbolic representation despite being developed for different research purposes. Several structural attributes exhibit strongly skewed distributions, whereas musical content attributes present considerably greater diversity.

The attributes **Kod**, **Bas**, **LNS**, and **Pay** are dominated by a single value across all datasets. In particular, **Kod = 9**, **Bas = 96**, and **Pay = 1** account for more than 84% of all observations, while their frequencies exceed 90% in the SymbTr v3.0 dataset. This indicates that these attributes primarily represent structural or formatting information rather than musically discriminative characteristics.

In contrast, the pitch-related attributes (**Nota53**, **NotaAE**, **Koma53**, and **KomaAE**) exhibit substantially lower dominant frequencies (approximately 15–16%), suggesting a much richer distribution of symbolic pitch values. The most common pitches are **Re5 (D5)** and the corresponding microtonal value **327**, reflecting frequently occurring melodic tones while preserving considerable pitch diversity throughout the corpus.

The temporal attributes (**Sira** and **Offset**) have extremely small dominant frequencies (approximately 0.2–0.3%), indicating that sequence positions and temporal offsets are highly variable across musical pieces. Such distributions are expected because these attributes represent note ordering and temporal locations rather than categorical musical properties.

The **Ms** attribute shows similarly low dominant frequencies (approximately 8%), suggesting substantial variability in note durations and temporal values across the datasets. Likewise, **Payda** exhibits a broader distribution than **Pay**, indicating greater diversity in rhythmic subdivisions than in numerator values.

For the textual attribute (**Soz1**), the placeholder symbol (`"."`) frequently represents the absence of lyrics. To provide a more informative summary, the reported table displays the second most frequent value whenever `"."` is the dominant token. In all three datasets, **SAZ** emerges as the most frequent meaningful lyric-related token after excluding the placeholder from reporting.

Finally, **VelOn** appears only in the phrase-level datasets and remains constant (**96**) for every observation, indicating that velocity information is fixed in these collections and therefore contributes little discriminative information for statistical modeling.

Overall, these results confirm that the three datasets employ a common symbolic encoding scheme while exhibiting consistent musical characteristics. The observed distributions provide valuable insights for feature engineering, symbolic representation learning, machine learning applications, and generative AI models for Turkish makam music.

## Next Chapter

The next chapter, **Interactive Makam Following Demo**, presents an interactive demonstration for exploring Turkish makam music.

The demo enables users to follow symbolic musical data and interactively examine the generated musical content.